import pandas as pd
import numpy as np
import gc
import os
import random

import torch
import torch.nn as nn
import torch.optim as optim

import lightgbm as lgb

from torch.utils.data import DataLoader, TensorDataset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report
)

torch.set_num_threads(1)

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)



In [2]:
# download
from huggingface_hub import hf_hub_download

features = hf_hub_download(
    repo_id="hendriow/mh100k",
    filename= "mh100.parquet",
    repo_type ="dataset"
)

labels = hf_hub_download(
    repo_id = "hendriow/mh100k",
    filename="mh100-labels.csv",
    repo_type="dataset"
)
names = hf_hub_download(
    repo_id="hendriow/mh100k",
    filename="feature_names.csv",
    repo_type="dataset"
)



/Users/angela/miniforge3/envs/aqua311/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
MHfeat = pd.read_parquet(features)
MHLabel = pd.read_csv(labels)
MHNames = pd.read_csv(names)

In [ ]:
x_mh = MHfeat
y_mh = MHLabel["class"].astype(np.int64)

x_mh_train, x_mh_test, y_mh_train, y_mh_test = train_test_split(
    x_mh,
    y_mh,
    test_size =0.2,
    random_state=42,
    stratify=y_mh
)


x_mh_aqua_train, x_mh_val_a, y_mh_aqua_train, y_mh_val_a = train_test_split(
    x_mh_train,
    y_mh_train,
    test_size =0.2,
    random_state=42,
    stratify = y_mh_train
)

In [78]:
mh_needed = np.arange(len(x_mh))
mh_train_fs, mh_test_fs = train_test_split(
    mh_needed,
    test_size=0.2,
    random_state=42,
    stratify=y_mh
)
mh_aqua_train_fs, mh_val_fs = train_test_split(
    mh_train_fs,
    test_size=0.2,
    random_state=42,
    stratify=y_mh.iloc[mh_train_fs]
)

mh_test_sample_fs, _ = train_test_split(
    mh_test_fs,
    train_size=5000,
    random_state=42,
    stratify = y_mh.iloc[mh_test_fs]
)

mh_train_sample_fs, _ = train_test_split(
    mh_aqua_train_fs,
    train_size=20000,
    random_state=42,
    stratify=y_mh.iloc[mh_aqua_train_fs]

)

mh_val_sample_fs, _ = train_test_split(
    mh_val_fs,
    train_size=5000,
    random_state=42,
    stratify = y_mh.iloc[mh_val_fs]
)


Splitting 80/20

In [5]:
x_mh_train_sample, _, y_mh_train_sample, _ = train_test_split(
    x_mh_aqua_train,
    y_mh_aqua_train,
    train_size=20000,
    random_state=42,
    stratify=y_mh_aqua_train
)
x_mh_val_sample, _, y_mh_val_sample, _ = train_test_split(
    x_mh_val_a,
    y_mh_val_a,
    train_size=5000,
    random_state=42,
    stratify = y_mh_val_a
)
x_mh_test_sample, _, y_mh_test_sample, _ = train_test_split(
    x_mh_test,
    y_mh_test,
    train_size =5000,
    random_state=42,
    stratify=y_mh_test
)

In [6]:
print("Min feature values:", x_mh_train_sample.min().min())
print("Max featue value:", x_mh_train_sample.max().max())

print("training data:", x_mh_train_sample)

Min feature values: 0
Max featue value: 1
training data:        Permission::WAKE_LOCK  Permission::WRITE_EXTERNAL_STORAGE  \
34929                      1                                   1   
46911                      0                                   0   
29092                      0                                   0   
67045                      0                                   1   
77438                      0                                   0   
...                      ...                                 ...   
23608                      1                                   1   
52499                      1                                   1   
23925                      0                                   1   
51215                      1                                   1   
87159                      1                                   0   

       Permission::ACCESS_NETWORK_STATE  Permission::WRITE_SETTINGS  \
34929                                 1                

In [7]:
from sklearn.feature_selection import SelectKBest, chi2

selectmh = SelectKBest(score_func=chi2, k=256)
x_mh_train_kbest = selectmh.fit_transform(
    x_mh_train_sample,
    y_mh_train_sample

)
x_mh_val_kbest = selectmh.transform(x_mh_val_sample)
x_mh_test_kbest = selectmh.transform(x_mh_test_sample)

print(x_mh_train_kbest.shape)
print(x_mh_val_kbest.shape)
print(x_mh_test_kbest.shape)

(20000, 256)
(5000, 256)
(5000, 256)


scaling data - standard scalar

In [8]:
from sklearn.preprocessing import StandardScaler
scalemh = StandardScaler()

x_mh_train_scaled = scalemh.fit_transform(
    x_mh_train_kbest
).astype(np.float32)

x_mh_val_scaled = scalemh.transform(
    x_mh_val_kbest
).astype(np.float32)

x_mh_test_scaled = scalemh.transform(
    x_mh_test_kbest
).astype(np.float32)

y_mh_train_np = y_mh_train_sample.to_numpy(dtype=np.int64)
y_mh_val_np = y_mh_val_sample.to_numpy(dtype=np.int64)
y_mh_test_np = y_mh_test_sample.to_numpy(dtype=np.int64)



LGBM classifier on baseline data

In [9]:
mhlgb_before = lgb.LGBMClassifier(
    n_estimators = 20,
    learning_rate =0.05,
    num_leaves=7,
    max_depth=4,
    max_bin = 31,
    min_child_samples = 100,
    random_state=42,
    n_jobs=1,
    verbosity=1,
    force_col_wise=True,
    subsample_for_bin=1000

)

mhlgb_before.fit(
    x_mh_train_scaled,
    y_mh_train_np
)

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 1923, number of negative: 18077
[LightGBM] [Info] Total Bins 768
[LightGBM] [Info] Number of data points in the train set: 20000, number of used features: 256
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.096150 -> initscore=-2.240754
[LightGBM] [Info] Start training from score -2.240754


LGBMClassifier(force_col_wise=True, learning_rate=0.05, max_bin=31, max_depth=4,
               min_child_samples=100, n_estimators=20, n_jobs=1, num_leaves=7,
               random_state=42, subsample_for_bin=1000, verbosity=1)

In [10]:
y_mh_lgb_pred_b = mhlgb_before.predict(x_mh_test_scaled)
y_mh_lgb_proba_b = mhlgb_before.predict_proba(x_mh_test_scaled)[:,1]

print("before aqua")

print("accuracy:", accuracy_score(y_mh_test_np, y_mh_lgb_pred_b))
print("precision:", precision_score(y_mh_test_np,y_mh_lgb_pred_b))
print("recall", recall_score(y_mh_test_np, y_mh_lgb_pred_b))
print("f1 score:", f1_score(y_mh_test_np, y_mh_lgb_pred_b))
print("roc-auc:", roc_auc_score(y_mh_test_np, y_mh_lgb_proba_b))


before aqua
accuracy: 0.9658
precision: 0.8836633663366337
recall 0.7422037422037422
f1 score: 0.8067796610169492
roc-auc: 0.9581666964937602


Transformer before aqua

In [11]:
class FTTransformer_MH(nn.Module):
    def __init__(s, input_dim=256, d_model=8, num_classes=2):
        super().__init__()

        s.embedding = nn.Linear(16, d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=2,
            dim_feedforward=16,
            batch_first=True
        )

        s.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=1
        )

        s.classifier = nn.Sequential(
            nn.Linear(d_model * 16, 16),
            nn.ReLU(),
            nn.Linear(16, num_classes)
        )
    def forward(s,x):
        x = x.reshape(-1,16,16)
        x = s.embedding(x)
        x = s.transformer(x)
        x = x.reshape(x.size(0), -1)
        return s.classifier(x)
            

In [16]:
x_mh_tensor = torch.from_numpy(
   np.ascontiguousarray(x_mh_train_scaled, dtype = np.float32)
)
y_mh_tensor = torch.from_numpy(
   np.ascontiguousarray(y_mh_train_np, dtype = np.int64)
)
x_mh_val_tensor = torch.from_numpy(
    np.ascontiguousarray(x_mh_val_scaled, dtype = np.float32)
)
y_mh_val_tensor = torch.from_numpy(
    np.ascontiguousarray(y_mh_val_np, dtype = np.int64)
)

mh_training_load = DataLoader(
    TensorDataset(x_mh_tensor, y_mh_tensor),
    batch_size=256,
    shuffle=True
)



In [17]:
print(len(mh_training_load))
print(x_mh_val_tensor.shape)

79
torch.Size([5000, 256])


In [18]:
modelmh_before = FTTransformer_MH()
optimizer_mh = torch.optim.Adam(
    modelmh_before.parameters(),
    lr = 0.001
)
criterionmh = nn.CrossEntropyLoss()

[transformers] Disabling PyTorch because PyTorch >= 2.6 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [23]:

# train transform
epochs =2
for epoch in range(epochs):
    modelmh_before.train()
    tot_loss = 0

    for xb, yb in mh_training_load:
        optimizer_mh.zero_grad()
        output = modelmh_before(xb)
        loss = criterionmh(output, yb)

        loss.backward()
        optimizer_mh.step()
        tot_loss += loss.item()
    print(
        f"Epoch {epoch+1}/{epochs},"
        f"Loss: {tot_loss/len(mh_training_load):.2f}"
    )

Epoch 1/2,Loss: 0.19
Epoch 2/2,Loss: 0.13


In [25]:
x_mh_test_tenor = torch.from_numpy(
    np.ascontiguousarray(x_mh_test_scaled, dtype=np.float32)
)
y_mh_test_tenor = torch.from_numpy(
    np.ascontiguousarray(y_mh_test_np, dtype=np.int64)
)
print(x_mh_test_tenor.shape)
print(y_mh_test_tenor.shape)

torch.Size([5000, 256])
torch.Size([5000])


In [27]:
modelmh_before.eval()
with torch.no_grad():
    output_bef = modelmh_before(x_mh_test_tenor)
    probabilities_b = torch.softmax(
        output_bef, dim =1
    )[:,1].numpy()

    predictions_b = torch.argmax(
        output_bef, dim=1
    ).numpy()
y_true_b = y_mh_test_tenor.numpy()

accuracy_mh_b = accuracy_score(y_true_b, predictions_b)

precision_mh_b = precision_score(y_true_b, predictions_b, zero_division=0)
recall_mh_b = recall_score(y_true_b, predictions_b, zero_division=0)
f1_b = f1_score(y_true_b, predictions_b, zero_division=0)

roc_auc_b = roc_auc_score(
    y_true_b, probabilities_b
)

print("Accuracy:", accuracy_mh_b)
print("Precision:", precision_mh_b)
print("Recall:",recall_mh_b)
print("F1-score:",f1_b)
print("ROC-AUC:",roc_auc_b )



Accuracy: 0.9592
Precision: 0.784394250513347
Recall: 0.7941787941787942
F1-score: 0.7892561983471075
ROC-AUC: 0.9551227227704326


baseline data

In [30]:
transformer_mh_b_r = {
    "accuracy": accuracy_mh_b,
    "precision": precision_mh_b,
    "recall": recall_mh_b,
    "f1": f1_b,
    "roc-auc": roc_auc_b
    }
print(pd.Series(transformer_mh_b_r))

accuracy     0.959200
precision    0.784394
recall       0.794179
f1           0.789256
roc-auc      0.955123
dtype: float64


IMPORTING AQUA

In [31]:
print("AQuA imports:")
print("Aqdata:", "Aqdata" in globals())
print("AqNet:", "AqNet" in globals())
print("AUM:", "AUM" in globals())

print("\nTraining data:")
print("Features:", x_mh_train_scaled.shape)
print("Labels:", y_mh_train_np.shape)

print("\nValidation data:")
print("Features:", x_mh_val_scaled.shape)
print("Labels:", y_mh_val_np.shape)

print("\nPyTorch threads:", torch.get_num_threads())

AQuA imports:
Aqdata: False
AqNet: False
AUM: False

Training data:
Features: (20000, 256)
Labels: (20000,)

Validation data:
Features: (5000, 256)
Labels: (5000,)

PyTorch threads: 1


In [36]:
import sys
import importlib

AQUA_path = "/Users/angela/aqua"
if AQUA_path not in sys.path:
    sys.path.insert(0, AQUA_path)

importlib.invalidate_caches()

import aqua
from aqua.configs import main_config
from aqua.data import Aqdata
from aqua.models.presets import AqNet
from aqua.models.cleaning_modules.aum.aum import AUM
print("done")

[nltk_data] Downloading package punkt to /Users/angela/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/angela/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


done


In [38]:
DataQ_AQUA = Aqdata(
    x_mh_train_scaled.astype(np.float32),
    y_mh_train_np.astype(np.int64),
    noise_rate=0.0
)
print(DataQ_AQUA.data.shape)
print(DataQ_AQUA.labels.data.shape)

(20000, 256)
(20000,)


In [ ]:
class AUMCompatibleMLP_MH(nn.Module):
    def __init__(s, i_dim=256, o_dim=3):
        super().__init__()
        
        s.o_dim = o_dim
    
        s.network = nn.Sequential(
            nn.Linear(i_dim, 64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,o_dim)
        )
    def forward(s, x, kwargs=None):
        return s.network(x)



In [46]:
test_aqua_model = AUMCompatibleMLP_MH()

test_aqua_i = torch.from_numpy(
    np.ascontiguousarray(x_mh_test_scaled[:4], dtype=np.float32)
)
print(test_aqua_model(test_aqua_i).shape)

torch.Size([4, 3])


In [49]:
AUM_MH = AUMCompatibleMLP_MH(i_dim=256, o_dim=3)

optimizerMH = torch.optim.Adam(
    AUM_MH.parameters(),
    lr = 0.01
)
AqNetMH = AqNet(
    model = AUM_MH,
    output_dim = 3,
    epochs=2,
    batch_size=64,
    lr=0.01,
    device="cpu",
    optimizer=optimizerMH
)

AUM_MH = AUM(
    model=AqNetMH,
    optimizer=optimizerMH
)

In [53]:
np.random.seed(42)

AUM_m = AUM_MH.find_label_issues(
    DataQ_AQUA,
    alpha=0.05
)

Epoch: 1/2: 100%|██████████| 313/313 [00:02<00:00, 133.69it/s, loss=0.713 (0.730)]
INFO:root:


Epoch: 2/2: 100%|██████████| 313/313 [00:02<00:00, 149.05it/s, loss=0.678 (0.729)]
INFO:root:


Epoch: 1/2: 100%|██████████| 313/313 [00:01<00:00, 168.05it/s, loss=0.686 (0.700)]
INFO:root:


Epoch: 2/2: 100%|██████████| 313/313 [00:01<00:00, 160.91it/s, loss=0.746 (0.687)]
INFO:root:




In [54]:
print("Flagged:", AUM_m.sum())
print("Training samples:", len(AUM_m))
print("Remaining samples:", (~AUM_m.sum()))

Flagged: 1042
Training samples: 20000
Remaining samples: -1043


In [55]:
CleanedmH = ~AUM_m

x_mh_train_clean = x_mh_train_scaled[CleanedmH]
y_mh_train_clean = y_mh_train_np[CleanedmH]

print("Training Features:", x_mh_train_clean.shape)
print("Training labels:", y_mh_train_clean.shape)
print("Counts:", np.unique(y_mh_train_clean, return_counts=True))

Training Features: (18958, 256)
Training labels: (18958,)
Counts: (array([0, 1]), array([17149,  1809]))


In [56]:
mhlgb_after = lgb.LGBMClassifier(
    n_estimators = 20,
    learning_rate =0.05,
    num_leaves=7,
    max_depth=4,
    max_bin = 31,
    min_child_samples = 100,
    random_state=42,
    n_jobs=1,
    verbosity=1,
    force_col_wise=True,
    subsample_for_bin=1000

)

mhlgb_after.fit(
    x_mh_train_clean,
    y_mh_train_clean
)

[LightGBM] [Warning] Using too small ``bin_construct_sample_cnt`` may encounter unexpected errors and poor accuracy.
[LightGBM] [Info] Number of positive: 1809, number of negative: 17149
[LightGBM] [Info] Total Bins 768
[LightGBM] [Info] Number of data points in the train set: 18958, number of used features: 256
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.095421 -> initscore=-2.249166
[LightGBM] [Info] Start training from score -2.249166


LGBMClassifier(force_col_wise=True, learning_rate=0.05, max_bin=31, max_depth=4,
               min_child_samples=100, n_estimators=20, n_jobs=1, num_leaves=7,
               random_state=42, subsample_for_bin=1000, verbosity=1)

In [58]:
from sklearn.metrics import(
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

predictions_after = mhlgb_after.predict(x_mh_test_scaled)

probabilities_after = mhlgb_after.predict_proba(
    x_mh_test_scaled
)[:,1]

accuracy_AFTER = accuracy_score(
    y_mh_test_np, predictions_after
)
precision_AFTER = precision_score(
    y_mh_test_np, predictions_after, zero_division=0
)
recall_AFTER = recall_score(
    y_mh_test_np, predictions_after,zero_division=0)
f1_AFTER = f1_score(
    y_mh_test_np, predictions_after, zero_division=0
)
roc_AUC_AFTER = roc_auc_score(
    y_mh_test_np, probabilities_after
)

print("Accuracy:", accuracy_AFTER)
print("Precision:", precision_AFTER)
print("Recall:", recall_AFTER)
print("F1-score:", f1_AFTER)
print("ROC-AUC:", roc_AUC_AFTER )


Accuracy: 0.966
Precision: 0.8858560794044665
Recall: 0.7422037422037422
F1-score: 0.8076923076923077
ROC-AUC: 0.9569397218213328


In [60]:
lgb_MH_after = {
    "accuracy": accuracy_AFTER,
    "precision": precision_AFTER,
    "recall": recall_AFTER,
    "f1": f1_AFTER,
    "roc-auc": roc_AUC_AFTER
    }
print(pd.Series(lgb_MH_after))

accuracy     0.966000
precision    0.885856
recall       0.742204
f1           0.807692
roc-auc      0.956940
dtype: float64


In [63]:
x_mh_clean_ten = torch.from_numpy(
    np.ascontiguousarray(x_mh_train_clean, dtype=np.float32)
)

y_mh_clean_ten = torch.from_numpy(
    np.ascontiguousarray(x_mh_train_clean, dtype=np.int64)
)
y_mh_clean_ten = torch.from_numpy(
    np.ascontiguousarray(y_mh_train_clean, dtype=np.int64)
)

mh_clean_Train = DataLoader(
    TensorDataset(x_mh_clean_ten, y_mh_clean_ten),
    batch_size=256,
    shuffle = True
)
print("Features:", x_mh_clean_ten.shape)
print("Labels:", y_mh_clean_ten.shape)
print("Batches", len(mh_clean_Train))

Features: torch.Size([18958, 256])
Labels: torch.Size([18958])
Batches 75


In [65]:
modelMH_AFTER = FTTransformer_MH()
optimizer_mh_after = torch.optim.Adam(
    modelMH_AFTER.parameters(),
    lr=0.001
)
criterionmh_after = nn.CrossEntropyLoss()


In [66]:
epochs = 2
for epoch in range(epochs):
    modelMH_AFTER.train()
    tot_loss = 0

    for xx, yy in mh_clean_Train:
        optimizer_mh_after.zero_grad()
        output = modelMH_AFTER(xx)
        loss = criterionmh_after(output, yy)

        loss.backward()
        optimizer_mh_after.step()

        tot_loss += loss.item()

    avg_loss = tot_loss/len(mh_training_load)

    print(
        f"Epoch {epoch+1}/{epochs},"
        f"Loss: {avg_loss:.4f}"
    )

Epoch 1/2,Loss: 0.1957
Epoch 2/2,Loss: 0.1226


In [72]:
modelMH_AFTER.eval()
with torch.no_grad():
    output_AFTER = modelMH_AFTER(x_mh_test_tenor)
    probabilities_after = torch.softmax(
        output_AFTER, dim=1
    )[:,1].numpy()

    predictions_after =(
        probabilities_after >=0.5
    ).astype(int)
acc_AFTER = accuracy_score(
    y_mh_test_np, predictions_after
)
prec_AFTER = precision_score(
    y_mh_test_np, predictions_after, zero_division=0
)
rec_AFTER = recall_score(
    y_mh_test_np, predictions_after, zero_division=0
)
f1_AFT = f1_score(
    y_mh_test_np, predictions_after, zero_division=0
)
roc_auc_aft = roc_auc_score(
    y_mh_test_np, probabilities_after
)

print("Accuracy:", acc_AFTER)
print("Precision:", prec_AFTER)
print("Recall:", rec_AFTER)
print("F1:", f1_AFT)
print("ROC AUC:", roc_auc_aft)



Accuracy: 0.9522
Precision: 0.72
Recall: 0.8232848232848233
F1: 0.7681862269641125
ROC AUC: 0.9579212555534752


In [109]:
split_mht = pd.concat([
pd.DataFrame({
    "row_id": mh_train_sample_fs,
    "dataset": "MH100K",
    "split": "train",
    "label": np.asarray(y_mh)[mh_train_sample_fs]
}),
pd.DataFrame({
    "row_id": mh_val_sample_fs,
    "dataset": "MH100K",
    "split": "validation",
    "label": np.asarray(y_mh)[mh_val_sample_fs]
}),
pd.DataFrame({
    "row_id": mh_test_sample_fs,
    "dataset":"MH100K",
    "split": "test",
    "label": np.asarray(y_mh)[mh_test_sample_fs]
}) 
], ignore_index=True)

split_mht.to_csv("mhtsplit.csv", index=False)




In [76]:
print("Test sample:", x_mh_test_sample.shape)
print("Test labels:", y_mh_test_sample.shape)

Test sample: (5000, 24833)
Test labels: (5000,)


In [84]:
lgb_before_p = mhlgb_before.predict_proba( x_mh_test_scaled)[:,1]
lgb_after_p = mhlgb_after.predict_proba(x_mh_test_scaled)[:,1]

scores_b = pd.DataFrame({
    "row_id": mh_test_sample_fs,
    "value": "baseline",
    "column": "LightGBM",
    "label": y_mh_test_np,
    "score": lgb_before_p,
    "threshold": 0.5
})

scores_a = pd.DataFrame({
    "row_id": mh_test_sample_fs,
    "value": "after_aqua",
    "column": "LightGBM",
    "label": y_mh_test_np,
    "score": lgb_before_p,
    "threshold": 0.5
})

scores_MHT = pd.concat(
    [scores_b, scores_a],
    ignore_index=True
)
scores_MHT.to_csv("scores.csv", index=False)


In [ ]:


transformer_scores_before= pd.DataFrame({
    "row_id": mh_test_sample_fs,
    "value": "baseline",
    "column": "Transformer",
    "label": y_mh_test_np,
    "score": probabilities_b,
    "threshold": 0.5
})

transformer_scores_after= pd.DataFrame({
    "row_id": mh_test_sample_fs,
    "value": "after_aqua",
    "column": "Transformer",
    "label": y_mh_test_np,
    "score": probabilities_after,
    "threshold": 0.5
})

scores_MHT = pd.concat(
    [scores_MHT, transformer_scores_before, transformer_scores_after],
    ignore_index=True
)
scores_MHT.to_csv("scores.csv", index=False)


In [108]:

error_d = scores_MHT[
    (scores_MHT["column"]=="LightGBM")&
    (scores_MHT["value"]=="after_aqua")
].copy()

error_d["prediction"]= (
    error_d["score"] >= error_d["threshold"]
).astype(int)

false_pos = error_d[
    (error_d["label"]==0)&
    (error_d["prediction"]==1)
].copy()

false_neg = error_d[
    (error_d["label"]==1)&
    (error_d["prediction"]==0)
].copy()

falsepos_sample = false_pos.sample(
    n=min(10, len(false_pos)),
    random_state=42
).copy()

falseneg_sample = false_neg.sample(
    n=min(10, len(false_neg)),
    random_state=42
).copy()

falsepos_sample["category"] = "false_positive"
falseneg_sample["category"] = "false_negative"

errors_MHT = pd.concat(
    [falsepos_sample, falseneg_sample],
    ignore_index=True
)

errors_MHT = errors_MHT[
    ["row_id", "value", "column", "category"]
]

errors_MHT.to_csv("mhterrors.csv", index=False)